# MicroHH processing chain
This notebook serves as a template to set up, run, post-process and visualise a MicroHH simulation using the `microhh_processing` package.

In [ ]:
%load_ext autoreload
%autoreload 2


from pathlib import Path

import ls2d
import matplotlib.pyplot as plt
import numpy as np
import ucat

import microhh_processing_chain.cscs as cscs
import microhh_processing_chain.enums as enums
import microhh_processing_chain.initialisation as init
import microhh_processing_chain.io as io
import microhh_processing_chain.postprocessing as post
import microhh_processing_chain.preprocessing as pre
import microhh_processing_chain.visualisation as vis

In [ ]:
# set logging level to info
import logging

logging.basicConfig(level=logging.INFO)

In [ ]:
# get the username of the current user
import getpass

user = getpass.getuser()

## Initialize case

In [ ]:
scratch_path = Path(f"/capstor/scratch/cscs/{user}")
store_path = Path(f"/capstor/store/cscs/empa/em05/{user}")

### Default methods: Scrips
The argument `name` must match the name of the case in `microhh/cases`. The case will be copied to the scratch path and the necessary scripts will be generated.

In [ ]:
case_suffix = "first_microhh_run_default"

case = init.initialise_case(
    name="ch4_plumes",
    suffix=case_suffix,
    scratch_path=scratch_path,
    method=enums.InitMethod.DEFAULT,
    init_from="/users/smeier/microhh/cases/ch4_plumes",
    overwrite=True,
)

Now you can jump to the pre-processing step.

### Interactive methods: Notebooks

In [ ]:
case_suffix = "first_microhh_run_interactive"

case = init.initialise_case(
    name="ch4_plumes",
    suffix=case_suffix,
    scratch_path=scratch_path,
    method=enums.InitMethod.INTERACTIVE,
)

In [ ]:
wind_speed = 1  # m/s
emission_height = 10  # m
surface_roughness = 0.1  # m

### Create grid

In [ ]:
# INITIALISE GRID
itot = 256
jtot = 128
ktot = 256

dx = dy = 5.0
dz = dx / 2

grid = ls2d.grid.Grid_equidist(kmax=ktot, dz0=dz)
grid.plot()

### Create initial and boundary conditions

In [ ]:
# INITIALISE PROFILES
th = pre.interpolate_to_model_profile(grid=grid, variable_value=290)
ch4 = pre.interpolate_to_model_profile(grid=grid)
v = pre.interpolate_to_model_profile(grid=grid)
u = pre.interpolate_to_model_profile(grid=grid, variable_value=wind_speed)

# Write NetCDF input file for MicroHH
init_profiles = {
    "z": grid.z,
    "th": th,
    "u": u,
    "v": v,
    "ch4": ch4,
}

io.save_case_input(case.name, case.scratch_path, init_profiles)

In [ ]:
ch4_vis = vis.CaseViewer(case)
figs = ch4_vis.plot_initial_conditions()

### Create `.ini` file

In [ ]:
# INITIALISE .INI FILE
strength = ucat.convert_mass_per_time_unit(100, "kg/h", "kg/s")  # in kg/s
sigma_horiz = dx / 4
sigma_z = sigma_horiz if emission_height > 1 else 0.1
endtime = 1 * 60**2
savetime = 2 * 60
sampletime = 60

th_grad = np.gradient(th, grid.z)

In [ ]:
# Modify .ini file:
settings = {
    "advec": {
        "swadvec": "2i62",
        "cflmax": 1.2,
        "fluxlimit_list": "ch4",
    },
    "boundary": {
        "swboundary": "surface",
        "mbcbot": "noslip",
        "mbctop": "freeslip",
        "sbcbot": "dirichlet",  # scalar values
        "sbctop": "neumann",  # gradient
        "ubot": None,
        "utop": None,
        "vbot": None,
        "vtop": None,
        "sbot": 0.0,
        "stop": 0.0,
        "sbot[th]": th[0],
        "stop[th]": th_grad[-1],
        "z0m": surface_roughness,
        "z0h": surface_roughness,
        "swconstantz0": True,
        "swtimedep": False,
        "timedeplist": None,
        "scalar_outflow": "ch4",
        "flow_direction[north]": "outflow",
        "flow_direction[east]": "outflow",
        "flow_direction[south]": "outflow",
        "flow_direction[west]": "inflow",
    },
    "cross": {"swcross": True, "sampletime": sampletime, "crosslist": ["ch4_path"]},
    "dump": {
        "swdump": True,
        "dumplist": ["u", "v", "w", "ch4"],
        "sampletime": sampletime,
    },
    "land_surface": {},
    "buffer": {
        "swbuffer": True,
        "swupdate": True,
        "zstart": grid.zsize * 0.75,
        "sigma": 0.00223,
        "beta": 2.0,
    },
    "diff": {"swdiff": "smag2", "dnmax": 0.4},
    "fields": {
        "slist": "ch4",
        "visc": 1.0e-5,
        "svisc": 1.0e-5,
        "rndseed": 2,
        "rndamp[th]": 0.1,
        "rndz": grid.zsize * 0.5,
        "rndexp": 2.0,
    },
    "force": {
        "swlspres": False,
        "swls": False,
        "lslist": None,
        "swtimedep_ls": False,
        "timedeplist_ls": False,
        "swwls": False,
        "swwls_mom": False,
        "swtimedep_wls": False,
        "swnudge": False,
        "nudgelist": None,
        "scalednudgelist": None,
        "swtimedep_nudge": False,
        "timedeplist_nudge": None,
    },
    "grid": {
        "itot": itot,
        "jtot": jtot,
        "ktot": ktot,
        "xsize": itot * dx,
        "ysize": jtot * dy,
        "zsize": grid.zsize,
        "swspatialorder": 2,
    },
    "limiter": {
        "limitlist": "ch4",
    },
    "master": {
        "npx": 16,
        "npy": 8,
    },
    "source": {
        "swsource": True,
        "sourcelist": ["ch4"],
        "source_x0": [int(20 * dx)],
        "source_y0": [(jtot * dy) / 2],
        "source_z0": [emission_height],
        "sigma_x": [sigma_horiz],
        "sigma_y": [sigma_horiz],
        "sigma_z": [sigma_z],
        "strength": [strength],
        "swvmr": False,
        "swtimedep_strength": False,
    },
    "stats": {
        "swstats": True,
        "sampletime": sampletime,
    },
    "thermo": {
        "swthermo": "dry",
        "swbasestate": "anelastic",
        "pbot": 100000.0,
    },
    "time": {
        "starttime": 0,
        "endtime": endtime,
        "savetime": savetime,
        "adaptivestep": True,
        "dtmax": 60.0,
        "dt": 6.0,
        "rkorder": 4,
        "outputiter": 5,
    },
}

case.overwrite_ini(settings)

The `.ini` file can be inspected in the following way:

In [ ]:
case.open_mhh_in_vscode("ini")

The `ini` settings can also be modified after initialisation

In [ ]:
case.modify_ini({"diff": {"cs": 0.23}})

The `ini` entries can be accessed from `case` in different ways, e.g.:

In [ ]:
case.config.grid.itot

In [ ]:
case.config["grid"]["itot"]

In [ ]:
case.config.get("grid").get("itot")

You can compare the .ini file settings of different cases using the `compare_ini` function:

In [ ]:
example_case_path = Path("../../../microhh/cases/jaenschwalde")
init.compare_ini(example_case_path, case.directory)

## Pre-process case

### Create bash script and run checks to validate the setup

In [ ]:
# DEFINE ALPS SETUP
alps_setup = cscs.AlpsSetup(
    cluster="eiger",
    account="em05",
    processor="cpu",
    run_time="00:30:00",
    partition="debug",
    ntasks_per_core=1,
    ntasks_per_node=128,
    uenv="prgenv-gnu/24.11:v2",
    view="default",
)

# DO PRE-PROCESSING
pre_processing = pre.PreProcessing(
    case,
    alps_setup,
    microhh_executable_path=f"/users/{user}/microhh/build.mpi",
)
pre_processing.prepare_simulation(overwrite=True)

In [ ]:
case.open_mhh_in_vscode("sbash")

In [ ]:
pre_processing.start_simulation()

## Monitor your job

In [ ]:
cscs.jobs_in_queue(add_progress=True)[
    ["START_TIME", "TIME", "TIME_LEFT", "END_TIME", "PROGRESS (%)"]
]

In [ ]:
cscs.print_simulation_progress(case)

In [ ]:
cscs.get_sacct_info(case.job_id)

In [ ]:
case.calculate_benchmark()

## Post-process case

### Create the 3D output and combine the prognostic variables


In [ ]:
post_processing = post.PostProcessing(case)
post_processing.create_3d_output(
    vars=["u", "v", "w", "ch4"],
    sample_time=sampletime,
)
post_processing.create_2d_output(vars=["ch4_path"])

You can always check which files are present in the case directory using the `summarise_files` method of the `Case` class:

In [ ]:
case.summarise_files()

Combine all data into one dataset for easier handling and visualisation. This takes roughly 2 minutes. Alternatively, you can also load the variables separately from the 3D output files.

In [ ]:
dd_all = post_processing.handle_prognostic_variables(
    vars=["u", "v", "w", "th", "ch4"],
    filename="all_vars",
    interpolate=False,
)

In [ ]:
dd_all_3d = io.open_microhh_nc(case, "all_vars")
dd_ch4_path = io.open_microhh_nc(case, "ch4_path.xy")

### Compare progonostic variables with initial profiles

In [ ]:
ch4_vis = vis.CaseViewer(case)

In [ ]:
fig = ch4_vis.plot_prognostic_variables(
    variables=["u", "v", "th"], data=dd_all_3d, plot_format="line"
)

### Analyse the methane plume


In [ ]:
visualise = vis.CaseViewer(case)
fig = visualise.plot_side_views(dd_all_3d, "CH4", timestep=-1)

In [ ]:
%matplotlib widget
ui = vis.VariableViewer(
    dd_ch4_path, variable="ch4_path", cmap="magma", time_format="s", vmin=0
)
ui.display()

### Load and plot statistics data

In [ ]:
stats = io.open_statistics(case)
stats

In [ ]:
%matplotlib inline
plt.close("all")
figs = ch4_vis.plot_stats(["u", "th", "ch4", "tke", "b", "rhoref", "obuk", "ustar"])

### Analyse wind

In [ ]:
fig = ch4_vis.compare_wind_profiles(timestep=-1)

### Store your data

In [ ]:
case_directory = scratch_path / f"ch4_plumes.{case_suffix}"

case = init.read_case(
    case_directory=case_directory,
    scratch_path=scratch_path,
)

In [ ]:
post_processing = post.PostProcessing(case)
post_processing.zip_binary_files(file_pattern="ch4_path.xy.000")

In [ ]:
post_processing.archive_case_on_store(
    store_path=store_path,
    vars=["u", "v", "w", "ch4_path.xy"],
)
post_processing.move_pattern_to_store(pattern="ch4_path*.zip", store_path=store_path)

### Other useful functions

In [ ]:
figs = vis.plot_case_comparison(...)
figs = vis.plot_

post_processing.convert_microhh_grid_to_coords(...)
post_processing.convert_microhh_timestep_to_date(...)